In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor

class PostgresClient:
    def __init__(self):
        self.conn = psycopg2.connect(
            host="localhost",
            port=5432,
            dbname="analytics_db",
            user="admin",
            password="admin"
        )

    def run_query(self, query: str):
        try:
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(query)
                return cur.fetchall()
        except Exception as e:
            return {"error": str(e)}

if __name__ == "__main__":
    client = PostgresClient()
    
    query = """
    SELECT COUNT(*) as total_trades FROM trades;
    """
    
    result = client.run_query(query)
    print(result)

client = PostgresClient()
conn = client.conn
cursor = conn.cursor()

query = """
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""
cursor.execute(query)
rows = cursor.fetchall()
schema = {}

for table, column in rows:
    schema.setdefault(table, []).append(column)
cursor.close()
conn.close()

[RealDictRow([('total_trades', 50000)])]


In [2]:
schema

{'athletes': ['athlete_id', 'name', 'sport', 'team'],
 'events': ['athlete_id', 'event_date', 'performance_score'],
 'tokens': ['token_id', 'athlete_id', 'initial_price'],
 'trades': ['trade_id',
  'user_id',
  'token_id',
  'trade_date',
  'quantity',
  'price'],
 'users': ['user_id', 'signup_date', 'region']}

In [7]:
SYSTEM_PROMPT = """
You are a PostgreSQL expert.

Given a database schema and a user question, generate a valid SQL query.

Rules:
- Return ONLY raw SQL (no markdown, no backticks, no explanations)
- Use only the tables and columns provided in the schema
- Do not hallucinate columns or tables
- Prefer simple, correct queries over complex ones
- Only generate SELECT statements (no INSERT, UPDATE, DELETE, DROP)

- If a requested column is not in a table, you MUST join the correct table using appropriate keys
- Do NOT substitute or rename columns incorrectly
- Only use a column if it exists in that table
- If multiple tables are required, generate the correct JOIN
- When grouping by a column, ensure it belongs to the correct table

Output must be executable SQL.

Example:

Question: total trades by signup date

SQL:
SELECT u.signup_date, COUNT(*) AS total_trades
FROM trades t
JOIN users u ON t.user_id = u.user_id
GROUP BY u.signup_date;
"""

In [8]:
#GENERATE_SQL
import os
from openai import OpenAI
from dotenv import load_dotenv

from app.schema import get_schema

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

SYNONYMS = {
    "users": ["customers", "clients"],
    "trades": ["transactions"]
}

def format_schema(schema: dict) -> str:
    text = ""
    for table, cols in schema.items():
        text += f"\nTable: {table}\n"
        for col in cols:
            text += f"- {col}\n"
    return text

def clean_sql(response_text: str) -> str:
    return (
        response_text
        .replace("```sql", "")
        .replace("```", "")
        .strip()
    )

def matches(term: str, text: str) -> bool:
    return (
        term in text or
        term.rstrip("s") in text or
        (term + "s") in text
    )

In [21]:
question = "Total number of trades by signup date."
question = question.lower()
table_scores = {}

for table, cols in schema.items():
    # print (table,cols,"\n")
    table_name = table.lower()

    # include synonyms
    synonyms = SYNONYMS.get(table_name, [])
    all_terms = [table_name] + synonyms
    score = 0
    # print(all_terms)
    
    # Table name match
    if any(matches(term, question) for term in all_terms):
        score += 3

    # Column matches
    for col in cols:
        col_clean = col.lower().replace("_", " ")
        if matches(col_clean,question):
        # col.lower() in question:
            print(col)
            score += 1

    # store score if relevant
    if score > 0:
       table_scores[table] = score

# sort tables by score
sorted_tables = sorted(
    table_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

# keep top N tables (important)
TOP_K = 3
selected_tables = [t[0] for t in sorted_tables[:TOP_K]]

filtered = {t: schema[t] for t in selected_tables}

print("TABLE SCORES:", table_scores)
print("SELECTED TABLES:", selected_tables)

signup_date
TABLE SCORES: {'trades': 3, 'users': 1}
SELECTED TABLES: ['trades', 'users']


In [22]:
all_terms

['users', 'customers', 'clients']

In [23]:
sorted_tables

[('trades', 3), ('users', 1)]